In [1]:
import torch
from torch import nn

def crossrelated(X, K):
    h = X.size()[0]
    w = X.size()[1]
    n_h = K.size()[0]
    n_w = K.size()[1]
    Y = torch.zeros((h-n_h+1, w-n_w+1))
    for i in range(h-n_h+1):
        for j in range(w-n_w+1):
            Y[i, j] = (X[i:i+n_h, j:j+n_w]*K).sum()
    return Y

X = torch.ones((6, 8))
X[:, 2:6] = 0
K = torch.tensor([[1, -1]])
Y = crossrelated(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [2]:
x = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]])
k = torch.tensor([[0, 1], [2, 3]])
crossrelated(x, k)

tensor([[19., 25.],
        [37., 43.]])

In [3]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, X):
        return crossrelated(X, self.weight) + self.bias
net = Conv2D((2, 2))
net(X)

tensor([[2.6519, 1.7012, 0.0000, 0.0000, 0.0000, 0.9507, 2.6519],
        [2.6519, 1.7012, 0.0000, 0.0000, 0.0000, 0.9507, 2.6519],
        [2.6519, 1.7012, 0.0000, 0.0000, 0.0000, 0.9507, 2.6519],
        [2.6519, 1.7012, 0.0000, 0.0000, 0.0000, 0.9507, 2.6519],
        [2.6519, 1.7012, 0.0000, 0.0000, 0.0000, 0.9507, 2.6519]],
       grad_fn=<AddBackward0>)

In [4]:
crossrelated(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [ ]:
conv2d = nn.Conv2d(1, 1, (1, 2), bias = False)
X = X.reshape(1, 1, 6, 8)
Y = Y.reshape(1, 1, 6, 7)
print('X:', X, '\n', 'Y:', Y)
num_epochs = 10
for epoch in range(num_epochs):
    y_hat = conv2d(X)
    l = (y_hat - Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= 3e-2*conv2d.weight.grad
    if (epoch + 1)%2  == 0:
        print(f'epoch {epoch + 1}, loss {l.sum()}')
    

X: tensor([[[[1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.]]]]) 
 Y: tensor([[[[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.]]]])
epoch 2, loss 1.949318289756775
epoch 4, loss 0.8625978827476501
epoch 6, loss 0.3899974524974823
epoch 8, loss 0.17632631957530975
epoch 10, loss 0.07972099632024765
epoch 12, loss 0.03604364022612572
epoch 14, loss 0.016296114772558212
epoch 16, loss 0.0073678456246852875
epoch 18, loss 0.003331162966787815
epoch 20, loss 0.001506089698523283


In [26]:
conv2d.weight.data

tensor([[[[ 0.9870, -0.9870]]]])